# Clase 11: Visión Computacional en Tiempo Real - Conteo de Tráfico Local

En esta sesión, damos el salto de la **Clasificación** (decir qué hay en una imagen) a la **Detección y Rastreo** (localizar y seguir objetos en movimiento). Implementaremos un sistema de monitoreo de tráfico capaz de contar autos y motos en tiempo real.

---

## 1. El Salto Tecnológico: De CNN a YOLOv8

Hasta ahora, nuestras redes procesaban una imagen a la vez. En video (30+ FPS), esto es ineficiente. Utilizaremos **YOLO (You Only Look Once)**, una arquitectura diseñada para predecir todas las cajas de objetos y sus clases en una sola pasada por la red.

### Conceptos de Ingeniería:
- **Detección vs. Tracking:** La detección localiza el objeto en cada cuadro. El *Tracking* (Rastreo) le asigna un ID único para entender que el auto en el cuadro #1 es el mismo que en el cuadro #30.
- **Líneas de Cruce (Tripwires):** Definiremos coordenadas espaciales para contar un objeto solo cuando cruce un umbral específico.



In [ ]:
# BLOQUE 1: INSTALACIÓN Y CONFIGURACIÓN
# Instalamos las librerías necesarias para la detección y gestión de video
!pip install ultralytics supervision

import cv2
import torch
from ultralytics import YOLO
import supervision as sv
import numpy as np

# Verificación de Hardware (RTX 5070 Ti)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f">>> Procesando en: {device}")

## 2. Definición del Escenario de Tráfico

Para que el sistema sea útil en una ciudad como Coronel Oviedo, debemos definir **qué queremos contar** y **dónde**. 

**Instrucciones para el alumno:**
1. Carguen un video de tráfico (puede ser uno capturado por ustedes o un dataset público).
2. Identifiquen las coordenadas (X, Y) donde los vehículos cruzan el área de interés.

In [ ]:
# BLOQUE 2: CONFIGURACIÓN DE LA LÍNEA DE CONTEO

# Definimos el modelo pre-entrenado (YOLOv8 Nano por su eficiencia en video)
model = YOLO('yolov8n.pt')

# Definimos una línea virtual que cruza la calle (Ajustar según el video)
LINE_START = sv.Point(0, 400)
LINE_END = sv.Point(1280, 400)

# Inicializamos el contador y los anotadores visuales
line_zone = sv.LineZone(start=LINE_START, end=LINE_END)
line_zone_annotator = sv.LineZoneAnnotator(thickness=2, text_thickness=2, text_scale=0.8)
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=2, text_scale=0.8)

print(">>> Sistema de conteo configurado y listo para el stream de video.")

## 3. Procesamiento y Análisis en Tiempo Real

Este bucle procesa el video cuadro por cuadro, detecta vehículos, filtra las clases de interés (Autos y Motos) y gestiona el conteo.

In [ ]:
# BLOQUE 3: EL MOTOR DE VISIÓN

def ejecutar_conteo(video_path):
    # Generador de cuadros para no saturar la memoria RAM
    generator = sv.get_video_frames_generator(source_path=video_path)
    
    for frame in generator:
        # Inferencia de YOLOv8 en la GPU
        results = model(frame, device=device)[0]
        detections = sv.Detections.from_ultralytics(results)
        
        # FILTRO DE INGENIERÍA: 
        # ID 2 = 'car', ID 3 = 'motorcycle', ID 5 = 'bus', ID 7 = 'truck'
        clases_interes = [2, 3, 5, 7]
        detections = detections[np.isin(detections.class_id, clases_interes)]
        
        # Activamos el disparador de la línea de conteo
        line_zone.trigger(detections=detections)
        
        # Dibujamos las detecciones y la línea en el video
        annotated_frame = box_annotator.annotate(scene=frame.copy(), detections=detections)
        annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections)
        line_zone_annotator.annotate(annotated_frame, line_counter=line_zone)
        
        # Visualización en ventana (Presionar 'q' para salir)
        cv2.imshow("Terminal de Tráfico Inteligente - FCyT UNCA", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cv2.destroyAllWindows()
    print(f"--- REPORTE FINAL ---")
    print(f"Total Entrada: {line_zone.in_count}")
    print(f"Total Salida: {line_zone.out_count}")

# Instrucción: Los alumnos deben colocar la ruta de su video aquí
# ejecutar_conteo('trafico_video.mp4')

## 4. Desafío de Ingeniería de Tráfico

**Tarea para el grupo:**
1. **Ajuste de FPS:** Midan a cuántos cuadros por segundo corre su terminal. ¿Es suficiente para detectar una moto que viaja a 60 km/h?
2. **Análisis de Oclusión:** ¿Qué ocurre cuando un camión oculta a una moto? ¿Cómo afecta esto al conteo final?
3. **Propuesta:** ¿Cómo usarían estos datos para optimizar los semáforos de la rotonda de Coronel Oviedo?